In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.morphology import skeletonize
from skimage.measure import label, regionprops

from topostats.io import LoadScans
from topostats.plottingfuncs import Colormap

VMIN = -3
VMAX = 4
cmap = Colormap().get_cmap()

In [ ]:
file = Path("/Users/sylvi/topo_data/minicircle/output/processed/minicircle.topostats")
assert file.exists()

loadscans = LoadScans(img_paths=[file], config={"loading": {"channel": "dummy"}})
loadscans.get_data()
img_dict = loadscans.img_dict

image_data = img_dict["minicircle.topostats"]

image = image_data.image
assert image is not None

plt.imshow(image, cmap=cmap, vmin=VMIN, vmax=VMAX)
plt.show()


grain_crops = image_data.grain_crops
assert grain_crops is not None
for grain_index, grain_crop in grain_crops.items():
    print(f"grain: {grain_index}")
    pixel_to_nm_scaling = grain_crop.pixel_to_nm_scaling
    grain_image_crop = grain_crop.image
    plt.imshow(grain_image_crop, cmap=cmap, vmin=VMIN, vmax=VMAX)
    plt.title(f"Grain {grain_index}")
    plt.show()
    grain_mask_crop = grain_crop.mask[:, :, 1]
    plt.imshow(grain_mask_crop, cmap="gray")
    plt.title(f"Grain {grain_index} mask")
    plt.show()
    skeleton = grain_crop.skeleton
    print(type(skeleton))
    # turn all splined traces into a raster image of pixels
    ordered_trace_data = grain_crop.ordered_trace
    total_grain_spline_mask = np.zeros(grain_mask_crop.shape, dtype=bool)
    if ordered_trace_data is not None:
        if ordered_trace_data.molecule_data is not None:
            for molecule_id, molecule_data in ordered_trace_data.molecule_data.items():
                print(f"molecule: {molecule_id}")
                circular = molecule_data.circular
                molecule_splined_coords = molecule_data.splined_coords
                if circular:
                    # add the first point to the end of the list to close the loop
                    molecule_splined_coords = np.vstack([molecule_splined_coords, molecule_splined_coords[0]])
                assert molecule_splined_coords is not None
                molecule_spline_mask = np.zeros(grain_mask_crop.shape, dtype=bool)
                for i in range(len(molecule_splined_coords) - 1):
                    x0, y0 = molecule_splined_coords[i]
                    x1, y1 = molecule_splined_coords[i + 1]
                    # set the number of points to interpolate as being the euclidean distance between the two points,
                    # plus one.
                    num_points = int(np.linalg.norm([x1 - x0, y1 - y0])) + 2
                    xs = np.linspace(x0, x1, num_points).astype(int)
                    ys = np.linspace(y0, y1, num_points).astype(int)
                    molecule_spline_mask[xs, ys] = True
                total_grain_spline_mask = np.logical_or(total_grain_spline_mask, molecule_spline_mask)
            total_grain_spline_mask = skeletonize(total_grain_spline_mask)
            plt.imshow(total_grain_spline_mask, cmap="gray")
            plt.title(f"Grain {grain_index} skeleton")
            plt.show()

    # Calculate the hole areas
    mask_hole_areas: list[float] = []
    spline_hole_areas: list[float] = []
    # find the areas of the holes in the mask
    labelled_mask = label(np.logical_not(grain_mask_crop), connectivity=1)
    plt.imshow(labelled_mask)
    plt.show()
    for region_index, region in enumerate(regionprops(labelled_mask)):
        # ignore if the region touches the border of the image (background)
        if (
            region.bbox[0] == 0
            or region.bbox[1] == 0
            or region.bbox[2] == total_grain_spline_mask.shape[0]
            or region.bbox[3] == total_grain_spline_mask.shape[1]
        ):
            continue
        hole_area = region.area * (pixel_to_nm_scaling**2)
        mask_hole_areas.append(hole_area)
    # find the areas of the holes in the spline mask
    labelled_spline = label(np.logical_not(total_grain_spline_mask), connectivity=1)
    plt.imshow(labelled_spline)
    plt.show()
    for region_index, region in enumerate(regionprops(labelled_spline)):
        # ignore if the region touches the border of the image (background)
        if (
            region.bbox[0] == 0
            or region.bbox[1] == 0
            or region.bbox[2] == total_grain_spline_mask.shape[0]
            or region.bbox[3] == total_grain_spline_mask.shape[1]
        ):
            continue
        hole_area = region.area * (pixel_to_nm_scaling**2)
        spline_hole_areas.append(hole_area)
    
    print(f"Grain {grain_index} mask hole areas (nm^2): {mask_hole_areas}")
    print(f"Grain {grain_index} spline hole areas (nm^2): {spline_hole_areas}")
    
    
